# eCommerce Funnel Analysis Task
**Goal:** Calculate unique user conversion rates from View -> Intent -> Purchase.

### How to Use This Analysis
This notebook performs funnel analysis on eCommerce data.
1. Ensure the raw CSV file is in the `data/raw/` folder.
2. Update the file path in the data loading cell if needed.
3. Run all cells to generate processed CSVs in `data/processed/`.

In [16]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [17]:
# Defining only the columns we actually need for the funnel
needed_columns = [ 'event_type', 'user_id', 'brand', 'price']

# 2. Optimize the text columns into memory-saving categories
optimized_types = {
    'event_type': 'category',
    'brand': 'category'
}

# 3. Load the data using our optimizations and parse timestamps
df = pd.read_csv(
    "../data/raw/2019-Oct.csv",
    usecols=needed_columns,
    dtype=optimized_types
)
df.head()


,event_type,brand,price,user_id
0,view,shiseido,35.79,541312140
1,view,aqua,33.20,554748717
2,view,NaN,543.10,519107250
3,view,lenovo,251.74,550050854
4,view,apple,1081.98,535871217


## Step 1: Raw Event Frequencies
First, looking at the total number of actions taken across the platform. However, these are total events, not unique users.

In [5]:
event_count = df['event_type'].value_counts()
event_count

event_type
view        40779399
cart          926516
purchase      742849
Name: count, dtype: int64

## Step 2: Unique Users per Event
To build an accurate funnel, we calculate the distinct number of users at each stage. Check if 'purchase' > 'cart' to determine if direct checkout is present.

In [6]:
distinct_event_count = df.groupby('event_type')['user_id'].nunique()
distinct_event_count

event_type
cart         337117
purchase     347118
view        3022130
Name: user_id, dtype: int64

## Step 3: Funnel Stage Deduplication (The Intent Bucket)
**Conditional Intent Logic:**
- If `purchase_users > cart_users`: Direct checkout exists. Intent includes both cart and direct purchase users (deduplicated).
- If `purchase_users ≤ cart_users`: No direct checkout anomaly. Intent = cart users only.

This ensures a logical funnel where Intent ≥ Purchase users always.

In [7]:
# Calculate deduplicated intent users (cart + direct purchases, deduped)
intent_data = df[df['event_type'].isin(['cart','purchase'])]
true_intent_users = intent_data['user_id'].nunique()

# Check if direct purchases exist (purchase > cart)
if distinct_event_count['purchase'] > distinct_event_count.get('cart', 0):
    intent_users = true_intent_users  # Include direct purchases in intent
    print(f"Total users with intent (cart + direct purchases): {intent_users}")
else:
    intent_users = distinct_event_count.get('cart', 0)  # Use cart only
    print(f"Total users with intent (cart only): {intent_users}")



Total users with intent (cart + direct purchases): 481458


## Step 4: Dynamic Funnel Aggregation
Finally, assembling these deduplicated metrics into a clean, automated summary table for dashboard ingestion.

In [8]:
data = {
    "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
    "Unique_Users": [
        distinct_event_count['view'],      
        intent_users,                 
        distinct_event_count['purchase']   
    ]
}

funnel_df = pd.DataFrame(data)
funnel_df

,Funnel_Stage,Unique_Users
0,01_View,3022130
1,02_Intent,481458
2,03_Purchase,347118


## Step 5: Conversion and Drop-off Analysis
To fulfill the project requirements, we calculate the stage-to-stage conversion and drop-off rates using vectorized division and the `.shift()` method to align our rows.

**How `.shift()` Works:**
- `.shift(1)` moves each row DOWN by 1 position, creating NULL in the first row.
- Example: `[100, 50, 25].shift(1)` → `[NaN, 100, 50]`
- **Purpose:** Align each stage's users with the PREVIOUS stage for conversion calculation.
  - Row 1 (View): 100 / NaN → 100% (first stage baseline)
  - Row 2 (Intent): 50 / 100 → 50% (50% of viewers showed intent)
  - Row 3 (Purchase): 25 / 50 → 50% (50% of intent users converted to purchase)
- We use `.fillna(100)` to set the first stage to 100% (no prior stage to compare against).

In [9]:
funnel_df['conversion_rate%'] = (funnel_df['Unique_Users'] / funnel_df['Unique_Users'].shift(1)) * 100 
funnel_df['conversion_rate%'] = funnel_df['conversion_rate%'].fillna(100).round(2)
funnel_df['drop_off_rate%'] = 100 - funnel_df['conversion_rate%']  
funnel_df

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,3022130,100.00,0.00
1,02_Intent,481458,15.93,84.07
2,03_Purchase,347118,72.10,27.90


### Key Insights & Business Recommendations:
* **The Massive Top-of-Funnel Leak:** Historically, the vast majority of traffic bounces after seeing the product page. Refer to the output table below for this month's exact View-to-Intent drop-off rate. 
    * **Recommendation:** The marketing and UI teams should continuously monitor this drop-off. Consider A/B testing better product images or making the "Add to Cart" button more prominent.
* **The High-Intent Goldmine:** Users who proceed past the View stage and show purchase intent have a significant likelihood of completing a purchase. Check the Intent-to-Purchase conversion rate in the output table to understand checkout efficiency.

In [10]:
#Exporting the overall funnel
funnel_df.to_csv('../data/processed/01_overall_funnel_NOV.csv', index=False)

## Step 6: Scaling Analysis with Functions (DRY Principle)
To compare performance across different segments (e.g., brands), we wrap our funnel logic into a reusable Python function. This prevents code duplication and allows us to generate custom funnels dynamically.

**Key Features of `analyze_brand_funnel()`:**
- **Conditional Intent Logic:** Applies the same intent bucket rules as Step 3, but per brand:
  - If `purchase_users > cart_users` for the brand: Intent = cart + direct purchases (deduplicated).
  - If `purchase_users ≤ cart_users`: Intent = cart users only.
- **DRY Benefits:** Eliminates repetitive code for each brand analysis.
- **Output:** Returns a complete funnel DataFrame with conversion rates, ready for comparison.

In [ ]:
def analyze_brand_funnel(brand_name):
    """
    Analyzes the eCommerce funnel for a specific brand, applying conditional intent logic.
    
    Args:
        brand_name (str): The brand to analyze (e.g., 'apple', 'samsung').
    
    Returns:
        pd.DataFrame: Funnel metrics with conversion rates.
    """
    # 1. Isolate data for the specific brand
    brand_df = df[df['brand'] == brand_name]
    
    # 2. Get distinct user counts per event type for this brand
    distinct_counts = brand_df.groupby('event_type')['user_id'].nunique()
    
    # 3. Apply conditional intent logic (same as overall funnel)
    cart_count = distinct_counts.get('cart', 0)
    purchase_count = distinct_counts.get('purchase', 0)
    if purchase_count > cart_count:
        # Direct purchases exist: Include both cart and direct purchases in intent (deduplicated)
        intent_count = brand_df[brand_df['event_type'].isin(['cart', 'purchase'])]['user_id'].nunique()
    else:
        # No direct purchases anomaly: Intent = cart users only
        intent_count = cart_count
    
    # 4. Build the funnel DataFrame
    data = {
        "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
        "Unique_Users": [
            distinct_counts.get('view', 0),    
            intent_count,
            distinct_counts.get('purchase', 0) 
        ]
    }
    
    brand_funnel_df = pd.DataFrame(data)
    
    # 5. Calculate conversion and drop-off rates (using shift method)
    brand_funnel_df['conversion_rate%'] = (brand_funnel_df['Unique_Users'] / brand_funnel_df['Unique_Users'].shift(1)) * 100
    brand_funnel_df['conversion_rate%'] = brand_funnel_df['conversion_rate%'].fillna(100).round(2)
    brand_funnel_df['drop_off_rate%'] = 100 - brand_funnel_df['conversion_rate%']
    
    return brand_funnel_df


## Step 7: Brand Comparison (Apple vs. Samsung)
Testing our `analyze_brand_funnel()` function with two major brands to demonstrate per-brand conditional intent logic.

**Key Features:**
- **Per-Brand Intent Handling:** Each brand's funnel applies the same conditional rules as the overall analysis.
  - If a brand has `purchase > cart`: Intent includes direct purchases for accurate representation.
  - If `purchase ≤ cart`: Intent = cart users only.
- **Comparison Insights:** Compare View-to-Intent drop-off and Intent-to-Purchase conversion to identify strengths/weaknesses.

**Business Context:** Apple often has strong brand loyalty leading to higher conversions, while Samsung may excel in top-of-funnel attraction.

In [12]:
# Analyze Apple funnel
apple_funnel = analyze_brand_funnel('apple')
apple_funnel

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,729319,100.00,0.00
1,02_Intent,82063,11.25,88.75
2,03_Purchase,65593,79.93,20.07


In [13]:
# Analyze Samsung funnel
samsung_funnel = analyze_brand_funnel('samsung')
samsung_funnel

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,935464,100.00,0.00
1,02_Intent,119555,12.78,87.22
2,03_Purchase,94073,78.69,21.31


## Step 8: Automated Top 10 Brand Pipeline (Dashboard Prep)
This step scales brand analysis to the top 10 brands dynamically, creating a consolidated "Long Data" table for BI dashboards. It applies the same conditional intent logic per brand for accurate comparisons.

**How the Automated Pipeline Works:**
1. **Dynamic Brand Selection:** `df['brand'].value_counts().head(10)` identifies the top 10 brands by total event volume, ensuring we analyze the most impactful ones without hardcoding.
2. **Per-Brand Funnel Generation:** For each brand, calls `analyze_brand_funnel(brand)` which:
   - Filters data to that brand.
   - Counts unique users per event.
   - Applies conditional intent logic (cart + direct purchases if purchase > cart, else cart only).
   - Calculates conversion rates using `.shift()`.
3. **Data Consolidation:** 
   - Adds a `Brand` column to each funnel DataFrame for identification.
   - Appends each to `all_brand_funnels` list.
   - `pd.concat(all_brand_funnels, ignore_index=True)` stacks them into `master_brand_df` (30 rows: 3 stages × 10 brands).
4. **Export for Dashboard:** Saves as CSV in `data/processed/`, ready for BI tools like Power BI/Tableau.

**User Instructions for Export Customization:**
- **Default Path:** `../data/processed/02_brand_comparison_NOV.csv` (adjust month/year as needed).
- **To Customize:** Change the filename in the `.to_csv()` call, e.g.:
  - For November: `master_brand_df.to_csv('../data/processed/02_brand_comparison_NOV.csv', index=False)`
  - For a different directory: `master_brand_df.to_csv('path/to/your/file.csv', index=False)`
- **Why NOV?** Current data is November 2019; update the month in the filename for new datasets (e.g., `OCT` for October).
- **BI Ready:** Import the CSV into Power BI/Tableau for multi-brand funnel visualizations.

In [14]:
# Finding the top 10 brands dynamically based on who has the most rows
top_10_brands = df['brand'].value_counts().head(10).index.tolist()

# Creating an empty list to hold our tables
all_brand_funnels = []

# Loop through the top 10 brands
for brand in top_10_brands:
    # Run function
    temp_df = analyze_brand_funnel(brand)
    
    # Add a column so we know which brand
    temp_df.insert(0, 'Brand', brand) 
    
    # Add this brand's table to our list
    all_brand_funnels.append(temp_df)

# Stack all 10 tables on top of each other into one master dataframe
master_brand_df = pd.concat(all_brand_funnels, ignore_index=True)

# 6. Export the final master brand table to CSV!
master_brand_df.to_csv('../data/processed/02_brand_comparison_NOV.csv', index=False)

master_brand_df
# Display the master table to check our work

,Brand,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,samsung,01_View,935464,100.00,0.00
1,samsung,02_Intent,119555,12.78,87.22
2,samsung,03_Purchase,94073,78.69,21.31
3,apple,01_View,729319,100.00,0.00
4,apple,02_Intent,82063,11.25,88.75
5,apple,03_Purchase,65593,79.93,20.07
6,xiaomi,01_View,470832,100.00,0.00
7,xiaomi,02_Intent,43409,9.22,90.78
8,xiaomi,03_Purchase,35063,80.77,19.23
9,huawei,01_View,234061,100.00,0.00


## Step 9: Price Tier Segmentation
This step analyzes funnel performance across price brackets to understand price sensitivity. Prices are dynamically categorized into Low, Medium, and High tiers using quantiles, then funnels are generated for each tier.

**How Price Tier Segmentation Works:**
1. **Data Filtering:** `df = df[df['price'] > 0]` removes invalid prices (zero/negative) to avoid quantile errors.
2. **Quantile Categorization:** `pd.qcut(df['price'], q=3, labels=['Low', 'Medium', 'High'])` divides prices into 3 equal-sized bins (tertiles).
   - **Why qcut?** Ensures equal distribution; e.g., Low: bottom 33%, Medium: middle 33%, High: top 33%.
3. **Funnel Generation:** For each tier, calls `analyze_funnel_by_segment()` (same as brand analysis) to build funnels with conditional intent.
4. **Consolidation:** `pd.concat(all_price_funnels)` stacks tiers into one DataFrame for comparison.
5. **Export:** Saves as CSV for BI tools.

**Business Insights:** Compare tiers to see if higher prices lead to better/worse conversions (e.g., premium products may have loyal buyers).

**User Instructions for Export Customization:**
- **Default Path:** `../data/processed/03_price_funnel_NOV.csv` (adjust month/year as needed).
- **To Customize:** Change the filename in the `.to_csv()` call, e.g., for November: `..._NOV.csv`.
- **Why NOV?** Current data is November 2019; update the month in the filename for new datasets.

In [15]:
# Define function with strict column naming parameter
def analyze_funnel_by_segment(column_name, segment_value, final_col_name):
    # Filter dataframe for specific segment
    segment_df = df[df[column_name] == segment_value]
    
    # Calculate unique users per stage
    distinct_counts = segment_df.groupby('event_type')['user_id'].nunique()
    cart_count = distinct_counts.get('cart', 0)
    purchase_count = distinct_counts.get('purchase', 0)
    if purchase_count > cart_count:
        intent_count = segment_df[segment_df['event_type'].isin(['cart', 'purchase'])]['user_id'].nunique()
    else:
        intent_count = cart_count
    
    # Build dataframe using the exact final_col_name requested
    data = {
        final_col_name: segment_value,
        "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
        "Unique_Users": [
            distinct_counts.get('view', 0),    
            intent_count,
            distinct_counts.get('purchase', 0) 
        ]
    }
    funnel_df = pd.DataFrame(data)
    
    # Calculate conversion and drop-off metrics
    funnel_df['conversion_rate%'] = (funnel_df['Unique_Users'] / funnel_df['Unique_Users'].shift(1)) * 100
    funnel_df['conversion_rate%'] = funnel_df['conversion_rate%'].fillna(100).round(2)
    funnel_df['drop_off_rate%'] = 100 - funnel_df['conversion_rate%']
    
    return funnel_df


# --- PRICE TIER ANALYSIS ---
# Filter out zero or negative prices to avoid issues with quantiles
df = df[df['price'] > 0]

# Categorize prices into 3 tiers
df['price_tier'] = pd.qcut(df['price'], q=3, labels=['Low', 'Medium', 'High'])

all_price_funnels = []
for tier in ['Low', 'Medium', 'High']:
    # Execute function and force output column name to 'price_tier'
    temp_df = analyze_funnel_by_segment('price_tier', tier, 'price_tier')
    all_price_funnels.append(temp_df)

pd.concat(all_price_funnels).to_csv('../data/processed/03_price_funnel_NOV.csv', index=False)
